# 07 — Relatar avaliação após congelamento

## Objetivo
Executar ou apresentar a avaliação de um candidato identificado por freeze_id, depois da decisão de elegibilidade experimental.

## Entradas
Freeze imutável, promoção registrada e holdout interno vinculado ao manifesto.

## Saídas
`reports/experimental_v2/evaluations/<freeze_id>/`: logits, IDs, métricas, intervalo e confusion matrix.

## Permissões de dados
Somente este fluxo de avaliação abre o holdout. Não treina, calibra, seleciona ou promove. Final test independente continua indisponível.

## Reprodutibilidade
Inferência sem augmentations, classe/preprocess do freeze, bootstrap por grupo com seed 20260928 e 2.000 réplicas. Não repetir para selecionar resultado.

Protocolo: [experimental_protocol_v2.md](../docs/experimental_protocol_v2.md).
Os resultados anteriores estão catalogados como `legacy_v1`; não comparar seus scores diretamente aos v2.


In [ ]:
from pathlib import Path
import sys

# Bootstrap de importação; a resolução dos dados fica nos módulos compartilhados.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/experimental_protocol.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scripts.v2_artifacts import repository_root, read_json, run_id
ROOT = repository_root()

from scripts.v2_artifacts import identifier, verify_bundle
FREEZE_ID = None
EXECUTE_EVALUATION = False
DEVICE = "cpu"
report_dir = None
if FREEZE_ID:
    identifier(FREEZE_ID)
    report_dir = ROOT / "reports/experimental_v2/evaluations" / FREEZE_ID
if EXECUTE_EVALUATION:
    if FREEZE_ID is None:
        raise ValueError("freeze_id é obrigatório")
    from scripts.evaluate_frozen import evaluate_frozen
    report_dir = evaluate_frozen(ROOT, FREEZE_ID, DEVICE)
else:
    print("Avaliação desativada. Nenhum holdout foi lido nesta execução de revisão.")


## Interpretar sem adaptar o candidato
O script exige uma promoção elegível anterior e hashes correspondentes, salva um relatório novo e recusa sobrescrever a avaliação existente.
Se a avaliação falhar, a tentativa permanece registrada. Corrigir infraestrutura exige revisão explícita da tentativa; não apagar registros para tentar obter melhor resultado.

Este holdout é **interno e historicamente observado**. Seus intervalos não corrigem exposição prévia, ausência de patient_id ou mudança de população.
Um resultado decepcionante deve permanecer no relatório. Adaptar o sistema a ele abre outro ciclo de desenvolvimento e exige nova avaliação independente.


In [ ]:
if report_dir is not None and (report_dir / "evaluation_report.json").exists():
    verify_bundle(report_dir)
    report = read_json(report_dir / "evaluation_report.json")
    display(report["interval"])
    display(pd.DataFrame(report["metrics"]["per_class"]).T)
    from scripts.v2_metrics import plot_confusion
    plot_confusion(report["metrics"], "Holdout interno — candidato congelado"); plt.show()
else:
    print("Relatório pós-freeze ainda não disponível; nenhum resultado é presumido.")


## Conclusão
O relatório descreve o candidato congelado e conserva todos os resultados. Não há caminho de retorno automático para treino, seleção ou promoção.

## Artefatos produzidos
Predições e logits associados a image_id/lesion_id/group_id/checkpoint, métricas por classe, intervalo por lesão, ambiente e manifesto de hashes.

## Próxima etapa
Publicar as limitações e planejar avaliação externa/prospectiva independente, sem ajustar o mesmo candidato com este holdout.
